# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Dataset Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via the Croissant schema URL:

https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset with mlcroissant
dataset = mlc.Dataset(croissant_url)

# Access and print metadata (name and description)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Published: {metadata.datePublished}\nLicense: {metadata.license}\nVersion: {metadata.version}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Entities in Croissant datasets, including record sets, fields, columns, and others, are referenced using their `@id` in accordance with FAIR principles. Here we enumerate the available record sets and their associated field IDs for this dataset.

In [ ]:
# List all record set @ids
record_sets = dataset.metadata.record_set

if not record_sets:
    print("No record sets found. Please check the schema or contact the dataset provider.")
else:
    print(f"Record Sets (by @id):")
    for record_set in record_sets:
        print(f"- {record_set['@id']}")

    # For each record set, print the fields
    for record_set in record_sets:
        print(f"\nRecord Set: {record_set['@id']}")
        fields = record_set.get('field', [])
        if isinstance(fields, dict):  # single field
            fields = [fields]
        print("Fields (by @id):")
        for field in fields:
            print(f"  - {field['@id']} (name: {field.get('name', '')})")

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. Use the record set and field `@id`s found above.

In [ ]:
# Collect record set IDs
record_set_ids = []

# Generate @id list for record_sets
if dataset.metadata.record_set:
    for record_set in dataset.metadata.record_set:
        record_set_ids.append(record_set['@id'])
else:
    print('No record sets found in dataset metadata.')

# Dictionary of DataFrames per record set
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df

for rs_id, df in dataframes.items():
    print(f"Record Set {rs_id} - columns:")
    print(df.columns.tolist())
    print(df.head(), '\n')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Operations include removing outliers, transforming data distributions, or grouping by key attributes. All references use `@id` for record sets and fields.

In [ ]:
# Example EDA: Choose a record set and field using their @id
# For illustration, select the first available record set
if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]

    # Demonstrate numeric processing: identify numeric fields
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        print(f"Using numeric field for EDA: {numeric_field_id}")

        # Filtering - show top 5 records greater than threshold
        threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Grouping by a categorical field if available
        group_fields = [col for col in df.columns if pd.api.types.is_object_dtype(df[col])]
        if group_fields:
            group_field_id = group_fields[0]
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id}:")
            print(grouped_df.head())
    else:
        print('No numeric fields found for EDA operations.')
else:
    print('No dataframes loaded from record sets.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.
This section demonstrates simple plots using `matplotlib` and `seaborn`, referencing fields by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualization: Distribution of numeric field
if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        plt.figure(figsize=(8,5))
        sns.histplot(df[numeric_field_id], kde=True)
        plt.title(f"Distribution of field @id: {numeric_field_id}")
        plt.xlabel(numeric_field_id)
        plt.ylabel("Frequency")
        plt.show()

        # Visualize group vs numeric value
        group_fields = [col for col in df.columns if pd.api.types.is_object_dtype(df[col])]
        if group_fields:
            group_field_id = group_fields[0]
            plt.figure(figsize=(8,5))
            sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
            plt.title(f"Boxplot of {numeric_field_id} by {group_field_id} (@id)")
            plt.xticks(rotation=45)
            plt.show()
            
    else:
        print('No numeric fields available for visualization.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We successfully loaded the FAIR^2 dataset meta-information and tabular records using `mlcroissant`, referencing all entities by their `@id`.
- The record sets and fields were reviewed and extracted dynamically, making the notebook adaptable to schema changes.
- Exploratory Data Analysis and visualization were performed using numeric and categorical fields, which are referenced directly by their `@id`.
- This approach enables compliance with FAIR data principles, promotes reproducibility, and ensures correct referencing throughout the data workflow.